In [ ]:
pip install selenium webdriver-manager


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
import re  # Pour détecter un prix dans le texte

# Options pour éviter l'affichage du navigateur
chrome_options = Options()
chrome_options.add_argument("--headless")  # Mode sans affichage
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Chemin du WebDriver (il se met à jour automatiquement)
from webdriver_manager.chrome import ChromeDriverManager
service = Service(ChromeDriverManager().install())

# Lancer le navigateur Selenium
driver = webdriver.Chrome(service=service, options=chrome_options)

# Aller sur Avito
driver.get("https://www.avito.ma/")

# Attendre quelques secondes pour que la page charge complètement
time.sleep(5)

# Récupérer les annonces de voitures
car_listings = driver.find_elements(By.CLASS_NAME, "sc-b57yxx-1.eGVXZM")  

datas = []

for listing in car_listings:
    try:
        # Extraire le modèle
        model_element = listing.find_element(By.CLASS_NAME, "sc-1x0vz2r-0.iHApav")  
        model = model_element.text.strip() if model_element else "None"

        # Extraire la transmission (Manuelle / Automatique) et le prix
        gearbox = "Non spécifiée"
        span_elements = listing.find_elements(By.TAG_NAME, "span")  # Récupérer tous les <span>

        price = "None"
        for span in span_elements:
            text = span.text.strip()
            
            # Vérifier si le texte contient un prix (ex: "150 000 DH")
            if re.search(r'\d+\s?\d*\s?DH', text):  
                price = text
                break  # Stop dès qu'on trouve un prix valide
            
            # Vérifier si le texte contient "Manuelle" ou "Automatique"
            if "Manuelle" in text or "Automatique" in text or "manuelle" in text or  "automatique" in text:
                gearbox = text

        # Ajouter les données à la liste
        datas.append({
            'Modèle': model,
            'Prix': price,
            'Transmission': gearbox
        })

    except Exception as e:
        print("Erreur lors de l'extraction :", e)

# Fermer le navigateur
driver.quit()

# Afficher les données récupérées
if datas:
    for data in datas:
        print(data)
else:
    print("Aucune annonce trouvée.")
